# System 15 — Multi-Filter Portfolio System (FOREX · Crypto · B3)

A **portfolio-level** trading system backtested across three markets with one
shared account.  Unlike the single-strategy notebooks, the rules here are
*cross-asset* and live in `source.portfolio_backtest.PortfolioBacktester`.

**The system contract**

| Rule | Where |
|---|---|
| One clear **entry trigger** (EMA cross / Donchian / MACD) | `MultiFilterSystemStrategy.generate_signals` |
| **≥ 3 confirmation filters** — trend (EMA200), momentum (RSI), strength (ADX); optional Ichimoku / Bollinger / SuperTrend | same |
| A **separate volume filter** (`tick_vol / SMA`), independent of the 3 confirmations | same |
| **ATR stop** at `sl_atr_mult × ATR_entry` | `PortfolioBacktester` |
| **Optional take-profit** (`use_take_profit` → an optimisation knob) | `PortfolioBacktester` |
| Systematic, **filter-free exit** (opposite trigger) + ATR stop | `PortfolioBacktester` |
| **≤ 2 % account risk / trade** (`size = equity·risk_fraction / (sl_atr_mult·ATR)`) | `PortfolioBacktester` |
| **≤ 3 concurrent open trades** | `PortfolioBacktester` |
| **FOREX currency exclusion** — `EURUSD` open ⇒ no other pair with `EUR`/`USD` | `parse_currencies` + engine |

**Data engine.** The raw data is MetaTrader-5 **M1** (≈ 200–330 MB/file:
31 forex + 3 crypto + 23 B3).  All loading/resampling is pushed to **PySpark**
(`local[*]`): each M1 file is window-aggregated to the target timeframes and
written to a parquet cache keyed by the source mtime, so re-runs are instant
and the driver never holds a full M1 frame.  Backtests then fan out across
processes (`source.parallel`), one cell per `(group, timeframe)`.

**Scope / requirements**

- Needs `pyspark>=3.5` + a JDK (`pip install -r requirements.txt`; `java -version`).
- Timeframes: **swing + intraday** per market (see `CONFIG`).
- The expensive WFO / sensitivity run on a **representative asset subset**
  (`WFO_ASSETS`); the simple baseline backtest uses **all** assets.
- Committed **unexecuted** — run via Jupyter / VS Code (or
  `jupyter nbconvert --to notebook --execute --inplace`).
- The exit is filter-free but, like every notebook here, limited to the
  engine's fixed-at-entry SL/TP + opposite-signal contract; richer exits
  (trailing/Chandelier/Kumo) are out of scope for v1 — see the closing notes.

In [ ]:
import gc
import sys
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from source import (
    discover_datasets, cpu_count,
    PYSPARK_AVAILABLE, build_spark_grid, spark_eda_summary, get_spark,
    canonical_tf,
    MultiFilterSystemStrategy, MultiFilterSystemParams,
    PortfolioBacktester, run_portfolio, run_portfolio_grid,
    portfolio_metrics, consolidated_index, CONSOLIDATED_WEIGHTS,
    parse_currencies, calmar_ratio, rolling_annualized_sharpe,
    annual_profit_factor, random_trade_returns,
    portfolio_walk_forward, portfolio_parameter_sensitivity,
    overfitting_report,
    compute_metrics, plot_backtest_dashboard, plot_wfo_dashboard,
)

warnings.filterwarnings("ignore")
N_JOBS = "auto"
print(f"cpu_count={cpu_count()}  N_JOBS={N_JOBS!r}  PySpark={PYSPARK_AVAILABLE}")
if not PYSPARK_AVAILABLE:
    print("\n[!] PySpark not importable. Install it before running §1+:")
    print("    pip install -r requirements.txt   (pyspark>=3.5, pyarrow)")
    print("    and ensure a JDK is on PATH (java -version).")

In [ ]:
# ======================== CONFIG (edit me) ===========================
# Swing + intraday timeframes per market. Trim for a quicker pass.
GROUP_TIMEFRAMES = {
    "forex":  ["15min", "1h", "4h", "1D"],
    "crypto": ["15min", "1h", "4h", "1D"],
    "b3":     ["15min", "1h", "1D"],          # B3 is session-bound
}

# Representative, currency-overlapping subset for the expensive WFO /
# sensitivity sections (the baseline backtest still uses ALL assets).
WFO_ASSETS = {
    "forex":  ["EURUSD", "GBPUSD", "USDJPY", "USDCHF",
               "AUDUSD", "USDCAD", "EURJPY", "EURGBP"],
    "crypto": ["BTCUSD", "ETHUSD", "LTCUSD"],
    "b3":     ["PETR4", "PETR3", "ABEV3", "BBAS3", "ITSA4", "BBDC3"],
}

# EDA Spark scan is the one full-file pass; keep it to a sample by default.
EDA_ASSETS = {g: a[:3] for g, a in WFO_ASSETS.items()}

MAX_CONCURRENT   = 3
RISK_FRACTION    = 0.02          # ≤ 2 % account risk per trade
INITIAL_CAPITAL  = 100_000.0
SLIPPAGE_POINTS  = 0.0
CACHE_DIR        = REPO_ROOT / "data" / "_spark_cache"

# FAST_DEMO trims TFs + universe so the whole notebook finishes quickly while
# you sanity-check it; set False for the full run the spec asks for.
FAST_DEMO = False
if FAST_DEMO:
    GROUP_TIMEFRAMES = {"forex": ["4h", "1D"], "crypto": ["4h", "1D"],
                        "b3": ["1h", "1D"]}
    WFO_ASSETS = {g: a[:3] for g, a in WFO_ASSETS.items()}
    EDA_ASSETS = {g: a[:2] for g, a in WFO_ASSETS.items()}

print("Timeframes:", GROUP_TIMEFRAMES)
print("WFO subset:", {g: len(a) for g, a in WFO_ASSETS.items()})
print(f"Risk/trade ≤ {RISK_FRACTION:.0%}  ·  max concurrent = {MAX_CONCURRENT}"
      f"  ·  initial capital = {INITIAL_CAPITAL:,.0f}")

## 1. Exploratory Data Analysis

`discover_datasets` is metadata-only (no rows read).  The per-asset summary
(rows, span, gaps, dispersion) is computed **in Spark** over the raw M1 files
— distributed, never pulled into the driver.  By default the heavy scan is
limited to `EDA_ASSETS`; widen it for a full-universe audit.

In [ ]:
metas = discover_datasets(REPO_ROOT / "data")
meta_rows = []
for (asset, tf), m in sorted(metas.items()):
    meta_rows.append({"group": m.group or "root", "asset": asset,
                      "src_tf": tf, "start": m.start.date(),
                      "end": m.end.date(),
                      "size_MB": round(Path(m.path).stat().st_size / 1e6, 1)})
meta_df = pd.DataFrame(meta_rows)
print(f"{len(meta_df)} datasets across groups: "
      f"{sorted(meta_df['group'].unique())}")
display(meta_df.groupby('group').agg(
    n=('asset', 'count'), total_GB=('size_MB', lambda s: round(s.sum()/1e3, 2)),
    first=('start', 'min'), last=('end', 'max')))
display(meta_df.head(12))

In [ ]:
# Distributed EDA over the raw M1 files (Spark). One session reused below.
spark = get_spark() if PYSPARK_AVAILABLE else None

eda = []
if spark is not None:
    for (asset, _tf), m in metas.items():
        if (m.group in EDA_ASSETS) and (asset in EDA_ASSETS[m.group]):
            s = spark_eda_summary(spark, str(m.path))
            eda.append({"group": m.group, "asset": asset, **s})
eda_df = pd.DataFrame(eda)
if not eda_df.empty:
    eda_df["span_days"] = (pd.to_datetime(eda_df["end"])
                           - pd.to_datetime(eda_df["start"])).dt.days
    display(eda_df[["group", "asset", "rows_m1", "start", "end",
                    "span_days", "close_mean", "close_std",
                    "close_nulls", "zero_vol_bars"]])
else:
    print("PySpark unavailable — skipping the distributed EDA scan.")

In [ ]:
# Quick visual EDA: resample one sample asset/group to 1D via Spark.
from source.spark_loader import _resample_spark

if spark is not None and not eda_df.empty:
    fig, axes = plt.subplots(len(EDA_ASSETS), 3,
                             figsize=(16, 3.4 * len(EDA_ASSETS)))
    axes = np.atleast_2d(axes)
    for r, (g, alist) in enumerate(EDA_ASSETS.items()):
        asset = alist[0]
        m = next(mm for (a, _), mm in metas.items()
                 if a == asset and mm.group == g)
        d = _resample_spark(spark, str(m.path), "1D")
        ret = d["close"].pct_change().dropna()
        axes[r, 0].plot(d.index, d["close"], lw=.8); axes[r, 0].set_title(f"{g}/{asset} close (1D)")
        axes[r, 1].hist(ret, bins=60, color="steelblue"); axes[r, 1].set_title("daily returns")
        axes[r, 2].plot(ret.rolling(20).std(), color="crimson", lw=.8)
        axes[r, 2].set_title("20D rolling vol")
        for a_ in axes[r]:
            a_.grid(alpha=.3)
    fig.tight_layout(); plt.show()
else:
    print("Skipped (no Spark / no EDA frames).")

## 2. Trading Features

`build_spark_grid` window-aggregates **every** asset to each configured
timeframe and writes a parquet cache (keyed by source mtime — re-runs hit the
cache).  This is the single heavy ETL pass; afterwards every backtest reads
small parquet slices.

`MultiFilterSystemStrategy.compute_indicators(df, full=True)` materialises the
full indicator library used for entry signals, confirmation filters, trend /
momentum / volatility detection and the volume filter:

- **Trend/MA:** EMA(fast/slow/200), SMA, Ichimoku (Tenkan/Kijun/Senkou A·B),
  SuperTrend, Linear trend via EMA slope
- **Bands/volatility:** Bollinger (mid/upper/lower/width), ATR, Donchian,
  Keltner-style ATR envelope
- **Momentum/oscillators:** RSI (Wilder), MACD (line/signal/hist),
  Stochastic %K/%D, CCI, Williams %R, ROC
- **Strength/direction:** ADX, +DI/−DI
- **Volume:** OBV, intraday VWAP, `vol_ratio = tick_vol / SMA(tick_vol)`

In [ ]:
spark_grid = {}
if PYSPARK_AVAILABLE:
    spark_grid = build_spark_grid(
        REPO_ROOT / "data", GROUP_TIMEFRAMES,
        cache_dir=CACHE_DIR, spark=spark, progress=True)
    print("\nResampled grid (cells = group × tf):")
    for g, tfs in spark_grid.items():
        for tf, assets in tfs.items():
            print(f"  {g:6s} {tf:5s}  {len(assets)} assets")
else:
    print("PySpark unavailable — cannot build the resampled grid.")

# Derive the WFO subset grid by selecting assets from the same cache.
wfo_grid = {
    g: {tf: {a: ds for a, ds in assets.items() if a in WFO_ASSETS.get(g, [])}
        for tf, assets in tfs.items()}
    for g, tfs in spark_grid.items()
}

In [ ]:
# Indicator panel on one cached (group, asset, tf).
if spark_grid:
    g0 = next(iter(spark_grid))
    tf0 = next(iter(spark_grid[g0]))
    a0 = next(iter(spark_grid[g0][tf0]))
    panel = (MultiFilterSystemStrategy(MultiFilterSystemParams())
             .compute_indicators(spark_grid[g0][tf0][a0].load(), full=True))
    view = panel.tail(600)
    fig, ax = plt.subplots(5, 1, figsize=(15, 15), sharex=True)
    ax[0].plot(view.index, view["close"], "k", lw=.9, label="close")
    for c, ls in [("ema_fast", "-"), ("ema_slow", "-"), ("trend_ema", "--")]:
        ax[0].plot(view.index, view[c], ls, lw=.8, label=c)
    ax[0].fill_between(view.index, view["bb_lower"], view["bb_upper"],
                       color="grey", alpha=.15)
    ax[0].fill_between(view.index, view["senkou_a"], view["senkou_b"],
                       color="green", alpha=.12)
    ax[0].legend(ncol=4, fontsize=8); ax[0].set_title(f"{g0}/{a0} {tf0} — price + EMAs + BB + Kumo")
    ax[1].plot(view.index, view["rsi"], "purple", lw=.8); ax[1].axhline(50, color="grey", ls=":")
    ax[1].set_title("RSI")
    ax[2].plot(view.index, view["macd"], lw=.8, label="macd")
    ax[2].plot(view.index, view["macd_signal"], lw=.8, label="signal")
    ax[2].bar(view.index, view["macd_hist"], width=.02, alpha=.4); ax[2].legend(fontsize=8)
    ax[2].set_title("MACD")
    ax[3].plot(view.index, view["adx"], "brown", lw=.8); ax[3].axhline(20, color="grey", ls=":")
    ax[3].set_title("ADX")
    ax[4].plot(view.index, view["vol_ratio"], "teal", lw=.8); ax[4].axhline(1.2, color="grey", ls=":")
    ax[4].set_title("Volume ratio (tick_vol / SMA)")
    for a_ in ax:
        a_.grid(alpha=.3)
    fig.tight_layout(); plt.show()
    print("Indicator columns:", len(panel.columns))
else:
    print("No grid — run §2 with PySpark available.")

## 3. System Definition

**Entry** (long; short symmetric). The *trigger* fires **and all active
confirmations agree** **and** the separate volume filter passes:

1. **Trigger** — `entry_signal`: fast-EMA crosses above slow-EMA *(default)*,
   or close breaks the prior Donchian high, or MACD crosses above its signal.
2. **Confirmation 1 — trend:** `close > EMA(trend_ema)`.
3. **Confirmation 2 — momentum:** `RSI > rsi_mid`.
4. **Confirmation 3 — strength:** `ADX ≥ adx_min`.
5. *(optional confirmations: above Ichimoku Kumo / not over-extended past the
   Bollinger band / SuperTrend bullish — off by default, grid candidates.)*
6. **Separate volume filter:** `tick_vol / SMA(tick_vol) ≥ vol_ratio_min`.

**Risk & exit**

- **Stop loss:** `entry ∓ sl_atr_mult · ATR_entry` (ATR-multiple, mandatory).
- **Take profit:** *optional* `entry ± tp_atr_mult · ATR_entry`
  (`use_take_profit` — an explicit optimisation candidate, §4/§6).
- **Exit (systematic, no filters):** opposite trigger closes the position;
  otherwise the ATR stop / optional TP / session-end.

**Portfolio constraints** (`PortfolioBacktester`, one shared account)

- **Sizing:** `size = equity · risk_fraction / (sl_atr_mult · ATR_entry)` ⇒
  the risk to the stop is exactly `equity · risk_fraction` ≤ **2 %**.
- **≤ 3 concurrent** open trades across the whole portfolio.
- **FOREX currency exclusion:** while a pair is open, no other instrument
  sharing either currency may be opened (auto-enabled for `group="forex"`;
  off for crypto/B3, which still respect the concurrency cap).

In [ ]:
# Currency-exclusion demonstration.
demo = ["EURUSD", "USDJPY", "EURGBP", "GBPJPY", "XAUUSD", "AUDUSD"]
print("parse_currencies (group='forex'):")
for s in demo:
    print(f"  {s:7s} -> {sorted(parse_currencies(s, 'forex'))}")
print("\nIf EURUSD is open, these are BLOCKED (share EUR or USD):")
held = parse_currencies("EURUSD", "forex")
for s in demo:
    cc = parse_currencies(s, "forex")
    print(f"  {s:7s} {'BLOCKED' if cc & held else 'allowed'}")
print("\nNon-forex (e.g. group='crypto') -> currency exclusion OFF:",
      parse_currencies("BTCUSD", "crypto"))

base = MultiFilterSystemParams(risk_fraction=RISK_FRACTION)
print("\nDefault system params:")
for k, v in base.as_dict().items():
    print(f"  {k:22s} = {v}")

## 4. Default Parameters & Optimisation Grid

`DEFAULTS` — the baseline (B3 gets a 09–18 session window per repo
convention).  `PARAM_GRID` — what §6's WFO searches: which **trigger** to use,
which **filters/TP** to switch on, and the indicator periods / ATR multiples.
`SENSITIVITY_VARIATIONS` — the one-at-a-time sweep for §7.

The grid is built so the order constraint
`ema_fast < ema_slow < trend_ema` and `macd_fast < macd_slow` always hold;
`MultiFilterSystemParams.is_valid()` is the safety net (an invalid combo emits
zero signals and is scored `-inf`, never crashing the WFO product expansion).

In [ ]:
_common = dict(atr_period=14, risk_fraction=RISK_FRACTION)
DEFAULTS = {
    "forex":  MultiFilterSystemParams(**_common),
    "crypto": MultiFilterSystemParams(**_common),
    "b3":     MultiFilterSystemParams(session_start=9, session_end=18, **_common),
}

PARAM_GRID = {
    "entry_signal":   ["ema_cross", "donchian", "macd_cross"],
    "ema_fast":       [10, 20],
    "ema_slow":       [50, 100],
    "trend_ema":      [150, 200],
    "donchian_period":[20, 40],
    "adx_min":        [15.0, 25.0],
    "use_take_profit":[False, True],
    "tp_atr_mult":    [2.0, 3.0],
    "sl_atr_mult":    [1.5, 2.0, 3.0],
    "vol_ratio_min":  [1.0, 1.3],
}

SENSITIVITY_VARIATIONS = {
    "ema_fast":      [8, 10, 15, 20, 25],
    "ema_slow":      [40, 50, 75, 100, 125],
    "trend_ema":     [100, 150, 200, 250],
    "adx_min":       [10.0, 15.0, 20.0, 25.0, 30.0],
    "sl_atr_mult":   [1.0, 1.5, 2.0, 2.5, 3.0],
    "tp_atr_mult":   [1.5, 2.0, 3.0, 4.0],
    "vol_ratio_min": [0.8, 1.0, 1.2, 1.5, 2.0],
}

from itertools import product as _prod
_n = len(list(_prod(*PARAM_GRID.values())))
print(f"PARAM_GRID raw combinations: {_n:,} "
      f"(invalid ones are auto-scored -inf, not run to completion)")
print(f"WFO runs on the representative subset only: "
      f"{ {g: len(a) for g, a in WFO_ASSETS.items()} }")

## 5. Backtest — default parameters, one indicator combination

One portfolio per `(group, timeframe)` cell, **all assets**, default params,
run in parallel (each worker reads only its own parquet slices).  Then a full
metrics + plots report for a showcase cell: balance curve, drawdown, rolling
**annualised** Sharpe, **annualised** profit factor, random-trade return
distribution, and the concurrent-open-positions trace (showing the ≤ 3 cap).

In [ ]:
baseline = {}
if spark_grid:
    baseline = run_portfolio_grid(
        spark_grid, DEFAULTS, strategy_cls=MultiFilterSystemStrategy,
        default_params=DEFAULTS["forex"], max_concurrent=MAX_CONCURRENT,
        initial_capital=INITIAL_CAPITAL, slippage=SLIPPAGE_POINTS,
        n_jobs=N_JOBS, progress=True)
    rows = []
    for (g, tf), res in sorted(baseline.items()):
        m = portfolio_metrics(res)
        rows.append({"group": g, "tf": tf, "trades": m["num_trades"],
                     "total_pnl": round(m["total_pnl"], 1),
                     "win_rate": round(m["win_rate"], 3),
                     "profit_factor": round(m["profit_factor"], 2),
                     "max_dd": round(m["max_drawdown"], 1),
                     "sharpe_ann": round(m["sharpe_daily"], 2),
                     "calmar": round(m["calmar_ratio"], 2),
                     "cons_idx": round(m["consolidated_index"], 3)})
    baseline_tbl = pd.DataFrame(rows)
    display(baseline_tbl.sort_values("cons_idx", ascending=False))
else:
    print("No grid — §5 needs the §2 Spark cache.")

In [ ]:
# Full metrics for the best baseline cell.
def show_full_metrics(res, title):
    m = portfolio_metrics(res)
    order = ["num_trades", "total_pnl", "win_rate", "profit_factor",
             "expectancy", "avg_win", "avg_loss", "max_drawdown",
             "max_consec_losses", "sharpe_daily", "sharpe_per_trade",
             "calmar_ratio", "recovery_factor", "t_stat", "p_value",
             "consolidated_index"]
    tbl = pd.DataFrame([(k, m.get(k)) for k in order],
                       columns=["metric", "value"]).set_index("metric")
    print(title)
    display(tbl)
    return m

show_cell = None
if baseline:
    show_cell = max(baseline,
                    key=lambda k: portfolio_metrics(baseline[k])["consolidated_index"]
                    if not baseline[k].trades.empty else -np.inf)
    res = baseline[show_cell]
    if res.trades.empty:
        print("Best cell has no trades — widen the universe / loosen filters.")
    else:
        _ = show_full_metrics(res, f"Showcase cell: {show_cell}")

In [ ]:
if show_cell and not baseline[show_cell].trades.empty:
    res = baseline[show_cell]
    fig, ax = plt.subplots(3, 2, figsize=(16, 13))
    fig.suptitle(f"Multi-Filter Portfolio System — {show_cell}", fontsize=14)

    ax[0, 0].plot(res.balance.index, res.balance.values, color="darkgreen")
    ax[0, 0].axhline(INITIAL_CAPITAL, color="grey", ls="--")
    ax[0, 0].set_title("Account balance over time")

    dd = res.equity - res.equity.cummax()
    ax[0, 1].fill_between(dd.index, dd.values, 0, color="crimson", alpha=.4)
    ax[0, 1].set_title("Drawdown (points)")

    rs = rolling_annualized_sharpe(res.equity, window_days=90)
    ax[1, 0].plot(rs.index, rs.values, color="navy")
    ax[1, 0].axhline(0, color="grey", ls="--")
    ax[1, 0].set_title("Rolling annualised Sharpe (90D)")

    apf = annual_profit_factor(res.trades).replace([np.inf], np.nan)
    ax[1, 1].bar([str(i) for i in apf.index], apf.values, color="steelblue")
    ax[1, 1].axhline(1.0, color="red", ls="--")
    ax[1, 1].set_title("Annualised profit factor (per year)")

    rt = random_trade_returns(res.trades, n=200, seed=42)
    ax[2, 0].hist(rt, bins=40, color="slateblue", edgecolor="black")
    ax[2, 0].axvline(0, color="red", ls="--")
    ax[2, 0].set_title(f"Return distribution — {len(rt)} random trades")

    ax[2, 1].plot(res.open_positions.index, res.open_positions.values,
                   drawstyle="steps-post", color="darkorange")
    ax[2, 1].axhline(MAX_CONCURRENT, color="red", ls="--")
    ax[2, 1].set_title(f"Concurrent open trades (cap = {MAX_CONCURRENT})")
    for row in ax:
        for a_ in row:
            a_.grid(alpha=.3)
    fig.tight_layout(); plt.show()

    fig2 = plot_backtest_dashboard(res, title=f"Standard dashboard — {show_cell}")
    plt.show()

    m = portfolio_metrics(res)
    sig = ("statistically significant (p<0.05)"
           if (m["p_value"] == m["p_value"] and m["p_value"] < 0.05)
           else "NOT statistically significant at 5%")
    print(f"t-stat={m['t_stat']:.2f}  p-value={m['p_value']:.4f}  → {sig}")

## 6. Optimisation — Walk-Forward on the Consolidated Index

**Objective.** A single scalar (`consolidated_index`) blending the qualities
the spec asks to jointly maximise — best Sharpe, win rate and returns, smallest
drawdown, largest statistical significance:

```
index =  0.35·clip(annualised Sharpe, -3, 5)
       + 0.15·(win_rate − 0.5)·4
       + 0.25·clip(total_pnl / |max_drawdown|, -2, 5)   # drawdown-adjusted return
       + 0.10·tanh(profit_factor − 1)
       + 0.15·min(|t-stat|, 4)/4                          # statistical significance
```
(weights = `CONSOLIDATED_WEIGHTS`; `-inf` if < 10 trades or no usable Sharpe).

**Method — Walk-Forward.** `portfolio_walk_forward` cuts the assets' common
timeline into folds; on each fold's in-sample slice the **whole `PARAM_GRID`**
is searched *with the portfolio engine* (combos fan out across processes), the
best combo by the consolidated index is locked, then evaluated **once** on the
untouched out-of-sample slice.  OOS equity is stitched fold-to-fold.

In [ ]:
print("Consolidated-index weights:", CONSOLIDATED_WEIGHTS)
wfo_results = {}
if wfo_grid:
    for g, tfs in wfo_grid.items():
        for tf, assets in tfs.items():
            if not assets:
                continue
            print(f"\n=== WFO {g}/{tf}  ({len(assets)} assets) ===")
            wfo_results[(g, tf)] = portfolio_walk_forward(
                assets, PARAM_GRID,
                strategy_cls=MultiFilterSystemStrategy,
                params_cls=MultiFilterSystemParams,
                group=g, n_splits=5, oos_ratio=0.25,
                score_fn=consolidated_index,
                max_concurrent=MAX_CONCURRENT,
                initial_capital=INITIAL_CAPITAL,
                n_jobs=N_JOBS, progress=True)
            gc.collect()
else:
    print("No WFO grid — run §2 with PySpark available.")

In [ ]:
wfo_rank = []
for (g, tf), w in wfo_results.items():
    if w.windows.empty:
        continue
    oos_cons = (consolidated_index(compute_metrics(
        type("R", (), {"trades": w.oos_trades, "equity": w.oos_equity})()))
        if not w.oos_trades.empty else float("nan"))
    wfo_rank.append({"group": g, "tf": tf, "folds": len(w.windows),
                     "oos_trades": len(w.oos_trades),
                     "mean_oos_sharpe": round(
                         pd.to_numeric(w.windows["oos_sharpe"],
                                       errors="coerce").mean(), 3),
                     "mean_degradation": round(
                         pd.to_numeric(w.windows["degradation_ratio"],
                                       errors="coerce").mean(), 3),
                     "oos_consolidated": round(oos_cons, 3)})
if wfo_rank:
    rank_df = pd.DataFrame(wfo_rank).sort_values(
        "oos_consolidated", ascending=False)
    display(rank_df)
    best_key = (rank_df.iloc[0]["group"], rank_df.iloc[0]["tf"])
    print("Best (group, tf) by OOS consolidated index:", best_key)
else:
    best_key = None
    print("No WFO windows produced (try FAST_DEMO=False data / wider grid).")

In [ ]:
if best_key and not wfo_results[best_key].windows.empty:
    w = wfo_results[best_key]
    display(w.windows[[c for c in w.windows.columns
                       if not c.startswith("param_")]])
    pcols = [c for c in w.windows.columns if c.startswith("param_")]
    print("WFO-picked parameters per fold:")
    display(w.windows[["fold"] + pcols])
    if not w.oos_trades.empty:
        base_eq = (baseline[best_key].equity
                   if best_key in baseline else None)
        fig = plot_wfo_dashboard(w, full_equity=base_eq)
        fig.suptitle(f"Portfolio WFO — {best_key}", y=1.02)
        plt.show()

## 7. Sensitivity & Overfitting Check

Sweep each parameter one-at-a-time around the WFO-selected configuration
(portfolio-scored, parallel).  `overfitting_report` then judges each knob:

- **cv** — coefficient of variation of the consolidated index across the sweep
  (how violently results move when the knob is nudged);
- **best_vs_neighbour** — relative drop from the best value to its better
  neighbour (a tall isolated spike ⇒ fragile / curve-fit);
- **plateau_frac** — fraction of swept values within 20 % of the best
  (a wide plateau ⇒ robust);
- **verdict** — `ROBUST` / `SENSITIVE` / `FRAGILE`, plus the mean WFO
  IS→OOS degradation as the overall overfitting flag.

In [ ]:
if best_key and not wfo_results[best_key].windows.empty:
    g, tf = best_key
    w = wfo_results[best_key]
    # most-frequently chosen params across folds = the configuration to probe
    pcols = [c for c in w.windows.columns if c.startswith("param_")]
    chosen = {c.replace("param_", ""):
              w.windows[c].mode().iloc[0] for c in pcols}
    base_kw = MultiFilterSystemParams(risk_fraction=RISK_FRACTION).as_dict()
    base_kw.update(chosen)
    if g == "b3":
        base_kw.update(session_start=9, session_end=18)
    probe_params = MultiFilterSystemParams(**base_kw)
    print("Probing configuration:", chosen)

    sens = portfolio_parameter_sensitivity(
        wfo_grid[g][tf], probe_params, SENSITIVITY_VARIATIONS,
        strategy_cls=MultiFilterSystemStrategy, group=g,
        max_concurrent=MAX_CONCURRENT, initial_capital=INITIAL_CAPITAL,
        n_jobs=N_JOBS)
    display(sens.pivot_table(index="value", columns="param",
                             values="consolidated_index"))

    rep = overfitting_report(sens, w.windows)
    display(rep)
    if "mean_oos_is_degradation" in rep.attrs:
        flag = rep.attrs.get("wfo_overfit_flag", False)
        print(f"Mean WFO IS→OOS degradation = "
              f"{rep.attrs['mean_oos_is_degradation']:.3f}  "
              f"→ {'⚠ OVERFIT RISK' if flag else 'acceptable'}")

    params_plt = list(SENSITIVITY_VARIATIONS)
    fig, ax = plt.subplots(2, 4, figsize=(20, 9))
    for i, pname in enumerate(params_plt):
        a_ = ax.flat[i]
        sub = sens[sens["param"] == pname].sort_values("value")
        a_.plot(sub["value"], sub["consolidated_index"], "o-")
        a_.set_title(pname); a_.grid(alpha=.3)
    for j in range(len(params_plt), len(ax.flat)):
        ax.flat[j].axis("off")
    fig.suptitle(f"Parameter sensitivity — consolidated index ({best_key})")
    fig.tight_layout(); plt.show()
else:
    print("No WFO selection to probe — run §6 first.")

In [ ]:
if spark is not None:
    spark.stop()
    print("Spark session stopped.")

## Takeaways & Implementation Notes

**What this notebook delivers**

- One **portfolio** system across FOREX / Crypto / B3 on a shared account,
  enforcing **≤ 2 % risk/trade**, **≤ 3 concurrent trades**, and **FOREX
  currency exclusion** — none expressible in the single-asset `Backtester`,
  hence the new `source.portfolio_backtest`.
- PySpark does all M1 → multi-timeframe ETL (parquet-cached); backtests fan
  out across processes — bounded RAM, every core used.
- A clear entry trigger + **3 confirmation filters + a separate volume
  filter**, ATR stop, **optional** TP (an explicit optimisation knob).
- A WFO optimiser maximising a single **consolidated index** (Sharpe + win
  rate + drawdown-adjusted return + profit factor + significance), plus a
  one-at-a-time **sensitivity / overfitting** verdict.

**Scope deviations (v1)**

- The exit is systematic but limited to the engine's *fixed-at-entry* SL/TP +
  opposite-signal contract. Trailing / Chandelier / Kumo-cross exits are
  **not** implemented (would need a per-bar dynamic-exit hook) — consistent
  with every other notebook in this repo.
- Optional confirmations (Ichimoku / Bollinger-extension / SuperTrend) are
  wired and grid-selectable but **off by default** so the 3 mandatory
  confirmations define the baseline.
- Named metals (`Platinum`, `Palladium`) expose no 3-letter code, so they
  only currency-exclude themselves (still subject to the concurrency cap).
- 1D Spark windows use UTC midnight boundaries (same convention as the repo's
  pandas resampler).
- Committed **unexecuted** — run it to populate outputs.